[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/niccoloridi/International-Law-and-AI/blob/main/IntlLaw_AI_Lab06_Pigeon_Vision.ipynb)

In [ ]:
#@title Lab 06 — Pigeon Vision
print("""
 █████             █████    █████                                                     █████████   █████
░░███             ░░███    ░░███                                         ███         ███░░░░░███ ░░███ 
 ░███  ████████   ███████   ░███         ██████   █████ ███ █████       ░███        ░███    ░███  ░███ 
 ░███ ░░███░░███ ░░░███░    ░███        ░░░░░███ ░░███ ░███░░███     ███████████    ░███████████  ░███ 
 ░███  ░███ ░███   ░███     ░███         ███████  ░███ ░███ ░███    ░░░░░███░░░     ░███░░░░░███  ░███ 
 ░███  ░███ ░███   ░███ ███ ░███      █ ███░░███  ░░███████████         ░███        ░███    ░███  ░███ 
 █████ ████ █████  ░░█████  ███████████░░████████  ░░████░████          ░░░         █████   █████ █████
░░░░░ ░░░░ ░░░░░    ░░░░░  ░░░░░░░░░░░  ░░░░░░░░    ░░░░ ░░░░                      ░░░░░   ░░░░░ ░░░░░ 
                                                                                                       
                                                                                                       
                                                                                                       

   Lab 06 // Pigeon Vision
   Melbourne Law Masters 2026
""")

*Brought to you by **Dr Niccolò Ridi**, Purveyor of Fine Scripts* – [KCL Profile](https://www.kcl.ac.uk/people/niccolo-ridi)

# Lab 06: Pigeon Vision – Object Detection and Autonomous Targeting

## Overview

In this lab, you will use a pre-trained YOLO object-detection model to:

1. **Classify and locate objects** in images and video
2. **Adjust confidence thresholds** and observe how bounding boxes, class labels, and probability scores change
3. **Understand the targeting pipeline** from pixels to decision – connecting computer vision to International Humanitarian Law
4. **Analyze error modes** (false positives, false negatives) and their human cost
5. **Reflect on meaningful human control** in autonomous systems

### Context: Lavender and the Targeting Pipeline

In 2024, investigative reporting revealed that the Israeli military used a machine learning system called **Lavender** to generate targeting recommendations in Gaza. According to these reports:

- The system scored individuals on a scale (likelihood of being a militant)
- A threshold was set: above it → recommend for airstrike
- A human operator had ~20 seconds to approve or reject each recommendation
- The volume of targets was such that meaningful individual review was reportedly impossible

This lab makes that pipeline **concrete and interactive**. You will not build a real targeting system–you will understand the logic of one, and grapple with the ethical and legal tensions embedded in that logic.

### What You'll Learn

- **Technical**: How YOLO works; what confidence scores mean; how thresholds affect detection
- **Legal**: IHL principles of distinction, proportionality, and precaution; the CCW debate on autonomous weapons
- **Ethical**: The difference between detection and targeting; the role of human judgment; error and civilian harm

## Where this lab sits on the AI map

Three broad paradigms, one rough map. This course uses all three.

| Paradigm | What it does | How | Example tools | This lab |
| --- | --- | --- | --- | --- |
| Symbolic | Follows explicit rules | Humans write logic; machine executes | IF-THEN rules, expert systems, treaty-as-code | Labs 01 (P1), 07 (P1) |
| **Subsymbolic (pattern recognition)** | Learns to classify, detect, or measure similarity | Neural network trained on labelled data; no explicit rules | CNNs (MobileNet, YOLO), BERT embeddings, sentence-transformers | Labs 01 (P2), 04, 06, 07 (P3), 09 (P3–4), 10 (P6) |
| Generative | Produces new text, image, or action | Large model predicts next token from a probability distribution | GPT, Gemini, Claude; agentic systems layer tools on top | Labs 02, 05, 09 (P1–2), 10 (P3–5) |

Generative models are technically subsymbolic too – they are neural networks under the hood. They are separated out because their behaviour (producing new content rather than classifying existing content) poses distinctive legal problems: authorship, attribution, hallucination, autonomy.

**This lab sits in:** **Subsymbolic**.

*Object detection is the classic discriminative task.*

In [ ]:
#@title Paper notes setup
#@markdown This lab will append your reflections to a markdown file you can download at the end of the session and use as material for your 5,000-word paper.

from pathlib import Path
from datetime import datetime

# Colab writes to /content; outside Colab we fall back to a dot-dir in HOME.
_notes_dir = Path("/content") if Path("/content").exists() else Path.home() / ".intllaw_ai_notes"
_notes_dir.mkdir(parents=True, exist_ok=True)
NOTES_PATH = _notes_dir / "paper_notes_lab_06.md"

if not NOTES_PATH.exists():
    NOTES_PATH.write_text(
        f"# Lab 06 – paper notes\n\nStarted: {datetime.now().isoformat(timespec='minutes')}\n\n"
    )

def _append_to_notes(heading, body):
    with NOTES_PATH.open("a") as f:
        f.write(f"\n## {heading}\n\n{body}\n")

print(f"Notes will accumulate in: {NOTES_PATH}")
print("At the end of the lab, open the Files panel (folder icon, left) to download.")

# Act 1 // Operation Politely Discourage Pigeons

A working pipeline. Real footage. YOLO finds birds. We define a region of interest – a windowsill, a balcony, a statue, a runway – and watch for a bird that lingers there. When one does, the system fires a "would spray" event: in deployment, that's a water sprinkler scaring the bird off.

This is opera-buffa applied computer vision. Detection → ROI check → lock-on → trigger. Nobody gets hurt; pigeons just get damp. Run it and watch.

In [ ]:
#@title 1) Install dependencies (YOLO + OpenCV + yt-dlp)
!pip -q install ultralytics opencv-python-headless yt-dlp

import os, cv2, math, shutil
from ultralytics import YOLO
print("✅ Installed")

In [ ]:
#@title 2) Set YouTube URL and download video (yt-dlp)
import os
import subprocess

YOUTUBE_URL = "https://www.youtube.com/watch?v=_aK8CyYBty0"  #@param {type:"string"}
OUT_DIR = "/content/pigeon_sim"
os.makedirs(OUT_DIR, exist_ok=True)

video_path = os.path.join(OUT_DIR, "video.mp4")

# yt-dlp can fail for several mundane reasons: the video was removed, the
# region is blocked, yt-dlp's signature-extraction code is out of date, the
# Colab IP is rate-limited. Guard the download so a failure prints a clear
# message and leaves video_path = None so downstream cells can check and
# short-circuit instead of crashing mid-pipeline.

download_cmd = [
    "yt-dlp",
    "-f", "bv*[ext=mp4]+ba[ext=m4a]/b[ext=mp4]/best",
    "-o", video_path,
    YOUTUBE_URL,
]

try:
    result = subprocess.run(download_cmd, capture_output=True, text=True, timeout=300)
    if result.returncode != 0 or not os.path.exists(video_path) or os.path.getsize(video_path) < 1024:
        raise RuntimeError(
            f"yt-dlp exited with code {result.returncode}.\n"
            f"--- stdout (tail) ---\n{result.stdout[-500:]}\n"
            f"--- stderr (tail) ---\n{result.stderr[-500:]}"
        )
    print("\u2705 Video at:", video_path)
    print(f"   size: {os.path.getsize(video_path):,} bytes")
except Exception as e:
    print(f"\u26a0\ufe0f  Could not download the YouTube video: {e}")
    print()
    print("   Act 1 needs a local video at /content/pigeon_sim/video.mp4. Options:")
    print("   1. Try a different YOUTUBE_URL in the cell above and re-run.")
    print("   2. Upload any short (30\u201360 s) mp4 to /content/pigeon_sim/video.mp4 via the")
    print("      Files panel and set video_path = '/content/pigeon_sim/video.mp4' by hand.")
    print("   3. Skip Act 1. Act 2 (Project Pigeon historical) and Act 3 (Lavender pivot")
    print("      on a street-scene image) do not depend on the YouTube download.")
    video_path = None

# Downstream cells should check: `if video_path is None: raise SystemExit(...)`
# Done conservatively via a flag the next few cells can inspect.
ACT_1_VIDEO_READY = video_path is not None and os.path.exists(video_path or "")

In [ ]:
#@title 3) Peek first frame to set ROI coordinates
import cv2


if not globals().get("ACT_1_VIDEO_READY", False):
    raise SystemExit("Skip this cell: Act 1 video download failed. See previous cell for fallback options.")

cap = cv2.VideoCapture(video_path)
ok, frame = cap.read()
cap.release()

if not ok:
    raise RuntimeError("Could not read first frame (try a different YouTube URL / format).")

h, w = frame.shape[:2]
print("✅ Frame size:", w, "x", h)

# Save a reference still so you can download + eyeball coordinates
ref_path = os.path.join(OUT_DIR, "first_frame.jpg")
cv2.imwrite(ref_path, frame)
print("✅ Saved:", ref_path)

In [ ]:
#@title 3.5) Trim video to a short clip (fast, no re-encode)
# Trim from START_SEC to END_SEC (e.g. 30 seconds total)

START_SEC = 2      #@param {type:"number"}
END_SEC   = 32     #@param {type:"number"}

trimmed_video_path = os.path.join(OUT_DIR, "video_trimmed.mp4")

# -ss before -i + -to + -c copy = fast stream cut (keyframe-accurate enough for prototyping)
!ffmpeg -y -ss {START_SEC} -to {END_SEC} -i "{video_path}" -c copy "{trimmed_video_path}"

print("✅ Trimmed video:", trimmed_video_path)
!ls -lh "{trimmed_video_path}"

In [ ]:
#@title 4) Config + ROI preview (ROI = Region Of Interest = landing zone; includes CONF_BIRD etc.)
# Edit the params (X/Y/W/H + thresholds), re-run this cell, and the preview updates.

import cv2, os, matplotlib.pyplot as plt

# --- Detection / logic params (used by Cell 5) ---
MODEL_NAME = "yolov8n.pt"   #@param {type:"string"}
CONF_BIRD = 0.45            #@param {type:"number"}  # try 0.25–0.60 depending on footage
LAND_N_FRAMES = 8           #@param {type:"integer"} # persistence for "landed"
MOVE_PX_THRESH = 10         #@param {type:"integer"} # centroid movement allowed per frame
SAVE_MAX_PER_EPISODE = 3    #@param {type:"integer"}

# --- Choose which video frame to preview ---
VIDEO_FOR_PREVIEW = trimmed_video_path  # expects Cell 3.5 to have run

# --- ROI A (LEFT landing zone): top-left + size ---
ROI_A_X = 20     #@param {type:"integer"}
ROI_A_Y = 100    #@param {type:"integer"}
ROI_A_W = 900    #@param {type:"integer"}
ROI_A_H = 900    #@param {type:"integer"}

# --- ROI B (RIGHT landing zone): top-left + size ---
ROI_B_X = 1000   #@param {type:"integer"}
ROI_B_Y = 20     #@param {type:"integer"}
ROI_B_W = 900    #@param {type:"integer"}
ROI_B_H = 900    #@param {type:"integer"}

def clamp_roi(x, y, w, h, W, H):
    x = max(0, min(x, W-1))
    y = max(0, min(y, H-1))
    w = max(1, min(w, W - x))
    h = max(1, min(h, H - y))
    return x, y, w, h

cap = cv2.VideoCapture(VIDEO_FOR_PREVIEW)
ok, frame = cap.read()
cap.release()
if not ok:
    raise RuntimeError("Could not read preview frame from video.")

H, W = frame.shape[:2]

# Clamp to frame bounds
ROI_A_Xc, ROI_A_Yc, ROI_A_Wc, ROI_A_Hc = clamp_roi(ROI_A_X, ROI_A_Y, ROI_A_W, ROI_A_H, W, H)
ROI_B_Xc, ROI_B_Yc, ROI_B_Wc, ROI_B_Hc = clamp_roi(ROI_B_X, ROI_B_Y, ROI_B_W, ROI_B_H, W, H)

ROI_A = (ROI_A_Xc, ROI_A_Yc, ROI_A_Xc + ROI_A_Wc, ROI_A_Yc + ROI_A_Hc)
ROI_B = (ROI_B_Xc, ROI_B_Yc, ROI_B_Xc + ROI_B_Wc, ROI_B_Yc + ROI_B_Hc)

def draw_roi(img, roi, label):
    x1, y1, x2, y2 = roi
    cv2.rectangle(img, (x1,y1), (x2,y2), (255,255,255), 3)
    cv2.putText(img, label, (x1, max(24, y1-8)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255,255,255), 2)

preview = frame.copy()
draw_roi(preview, ROI_A, f"ROI A (LEFT)  x={ROI_A_Xc}, y={ROI_A_Yc}, w={ROI_A_Wc}, h={ROI_A_Hc}")
draw_roi(preview, ROI_B, f"ROI B (RIGHT) x={ROI_B_Xc}, y={ROI_B_Yc}, w={ROI_B_Wc}, h={ROI_B_Hc}")

# Save + display
preview_path = os.path.join(OUT_DIR, "roi_preview.jpg")
cv2.imwrite(preview_path, preview)

print("Frame size:", W, "x", H)
print("MODEL_NAME:", MODEL_NAME)
print("CONF_BIRD:", CONF_BIRD, "| LAND_N_FRAMES:", LAND_N_FRAMES, "| MOVE_PX_THRESH:", MOVE_PX_THRESH)
print("ROI_A (x1,y1,x2,y2):", ROI_A)
print("ROI_B (x1,y1,x2,y2):", ROI_B)
print("Saved preview:", preview_path)

plt.figure(figsize=(10,7))
plt.imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("ROI preview — adjust ROI_* and thresholds then rerun")
plt.show()

In [ ]:
#@title 4.5a) Analyze bird confidences across the whole video (RESETS samples; suggests CONF_BIRD)
import os, shutil, cv2, numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

# --- Params ---
N_SAMPLES = 120          #@param {type:"integer"}
SAMPLE_MODE = "uniform"  #@param ["uniform","random"]
INFER_CONF = 0.10        #@param {type:"number"}
RANDOM_SEED = 0          #@param {type:"integer"}
MAX_EXAMPLE_IMAGES = 50  #@param {type:"integer"}

# --- Requirements ---
for var in ["MODEL_NAME","trimmed_video_path","OUT_DIR"]:
    if var not in globals():
        raise NameError(f"Missing `{var}`. Run earlier cells first.")

# --- Reset output folder on each run ---
examples_dir = os.path.join(OUT_DIR, "conf_bird_examples")
if os.path.exists(examples_dir):
    shutil.rmtree(examples_dir)
os.makedirs(examples_dir, exist_ok=True)

model = YOLO(MODEL_NAME)

cap = cv2.VideoCapture(trimmed_video_path)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {trimmed_video_path}")

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
print(f"✅ Video frames: {total_frames} | FPS: {fps:.2f}")

if total_frames <= 0:
    cap.release()
    raise RuntimeError("Could not read frame count. Try re-downloading or another clip.")

N = min(N_SAMPLES, total_frames)

# Choose frame indices
if SAMPLE_MODE == "uniform":
    idxs = np.linspace(0, total_frames - 1, N, dtype=int)
else:
    rng = np.random.default_rng(RANDOM_SEED)
    idxs = np.sort(rng.integers(0, total_frames, size=N))

bird_confs = []
saved_examples = 0

def read_frame_at(index: int):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(index))
    return cap.read()

for i, fidx in enumerate(idxs):
    ok, frame = read_frame_at(int(fidx))
    if not ok:
        continue

    res = model.predict(frame, imgsz=640, conf=INFER_CONF, verbose=False)[0]
    if res.boxes is None or len(res.boxes) == 0:
        continue

    bird_boxes = []
    for box in res.boxes:
        cls = int(box.cls[0])
        name = model.names.get(cls, str(cls))
        if name != "bird":
            continue
        conf = float(box.conf[0])
        bird_confs.append(conf)
        bird_boxes.append((box, conf))

    # save some annotated examples
    if bird_boxes and saved_examples < MAX_EXAMPLE_IMAGES:
        vis = frame.copy()
        for box, conf in bird_boxes[:3]:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            cv2.rectangle(vis, (x1,y1), (x2,y2), (255,255,255), 2)
            cv2.putText(vis, f"bird {conf:.2f}", (x1, max(15,y1-6)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
        ts = fidx / fps
        out = os.path.join(examples_dir, f"bird_conf_t{ts:.2f}_f{fidx}.jpg")
        cv2.imwrite(out, vis)
        saved_examples += 1

    # progress every 10 samples
    if (i + 1) % 10 == 0 or (i + 1) == len(idxs):
        print(f"…sampled {i+1}/{len(idxs)} frames | bird confs collected: {len(bird_confs)} | examples saved: {saved_examples}")

cap.release()

print("\n✅ Done sampling.")
print("Bird detections collected:", len(bird_confs))
print("Examples saved to:", examples_dir)

if not bird_confs:
    print("⚠️ No 'bird' detections found at INFER_CONF =", INFER_CONF)
    print("Try: INFER_CONF=0.05, a different clip, or MODEL_NAME='yolov8s.pt'.")
else:
    arr = np.array(bird_confs)
    q20 = float(np.quantile(arr, 0.20))
    q50 = float(np.quantile(arr, 0.50))
    q80 = float(np.quantile(arr, 0.80))
    suggested = q50  # default suggestion

    print("Bird conf min/median/max:", float(arr.min()), float(q50), float(arr.max()))
    print("Quantiles: q0.20=", round(q20,2), " q0.50=", round(q50,2), " q0.80=", round(q80,2))
    print(f"👉 Suggested CONF_BIRD (balanced) ≈ {suggested:.2f}")

    # Store suggestion so next cell can use it if you want
    CONF_BIRD_SUGGESTED = float(np.clip(suggested, 0.05, 0.99))

    plt.figure(figsize=(8,4))
    plt.hist(arr, bins=20)
    plt.axvline(CONF_BIRD_SUGGESTED, linestyle="--")
    plt.title("YOLO 'bird' confidence distribution (sampled across whole video)")
    plt.xlabel("confidence"); plt.ylabel("count")
    plt.show()

!ls -lh "{examples_dir}" | head -n 20

# Make conf list available to other cells (for q0.20/q0.80 selection)
globals()["bird_confs"] = bird_confs

In [ ]:
#@title 4.5b) Preview 4 examples + select how to SET CONF_BIRD
import os, glob, random
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# --- UI ---
SET_METHOD = "suggested (median)"  #@param ["suggested (median)","lenient (q0.20)","strict (q0.80)","manual"]
MANUAL_CONF_BIRD = 0.40            #@param {type:"number"}

# --- Requirements ---
if "OUT_DIR" not in globals():
    raise NameError("Missing OUT_DIR. Run earlier cells first.")
if "CONF_BIRD_SUGGESTED" not in globals():
    raise NameError("Missing CONF_BIRD_SUGGESTED. Run Cell 4.5a first.")
if "bird_confs" not in globals():
    # 4.5a currently doesn't expose bird_confs globally; we can still operate with suggested only.
    # But to support q0.20/q0.80 we need the distribution.
    print("⚠️ `bird_confs` not found globally. Lenient/strict quantiles won't be available unless you store them.")
    bird_confs = None

examples_dir = os.path.join(OUT_DIR, "conf_bird_examples")
paths = sorted(glob.glob(os.path.join(examples_dir, "*.jpg")))

if not paths:
    raise RuntimeError(f"No example images found in {examples_dir}. Run Cell 4.5a first.")

# Pick 4 images spaced across the list (more informative than random)
if len(paths) <= 4:
    chosen = paths
else:
    idxs = np.linspace(0, len(paths)-1, 4, dtype=int)
    chosen = [paths[i] for i in idxs]

# Display 4 examples
plt.figure(figsize=(14, 8))
for i, p in enumerate(chosen, start=1):
    img = Image.open(p)
    plt.subplot(2, 2, i)
    plt.imshow(img)
    plt.axis("off")
    plt.title(os.path.basename(p))
plt.suptitle("4 example frames with bird confidence labels (from 4.5a)", fontsize=14)
plt.show()

# Decide CONF_BIRD
def quantile_from_conf_list(q):
    arr = np.array(bird_confs, dtype=float)
    return float(np.quantile(arr, q))

if SET_METHOD == "suggested (median)":
    CONF_BIRD = float(CONF_BIRD_SUGGESTED)
    reason = "suggested median"
elif SET_METHOD == "manual":
    CONF_BIRD = float(MANUAL_CONF_BIRD)
    reason = "manual override"
elif SET_METHOD == "lenient (q0.20)":
    if not bird_confs:
        raise NameError("Need `bird_confs` to compute q0.20. See note below to store it in 4.5a.")
    CONF_BIRD = quantile_from_conf_list(0.20)
    reason = "q0.20 (lenient)"
elif SET_METHOD == "strict (q0.80)":
    if not bird_confs:
        raise NameError("Need `bird_confs` to compute q0.80. See note below to store it in 4.5a.")
    CONF_BIRD = quantile_from_conf_list(0.80)
    reason = "q0.80 (strict)"

# Clip to sane range
CONF_BIRD = float(np.clip(CONF_BIRD, 0.05, 0.99))
print(f"✅ CONF_BIRD set to {CONF_BIRD:.3f}  ({reason})")
print("Next: run Cell 5 again.")

In [ ]:
#@title 4.5c) Shifted confidence + landing parameters (based on 4.5a/4.5b results; overrideable)
import numpy as np
import cv2

# --- Requirements ---
for var in ["CONF_BIRD", "bird_confs", "trimmed_video_path"]:
    if var not in globals():
        raise NameError(f"Missing `{var}`. Run 4.5a then 4.5b first.")

# --- FPS (for LAND_SECONDS -> LAND_N_FRAMES) ---
cap = cv2.VideoCapture(trimmed_video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
cap.release()

# --- Suggested shifted-confidence defaults from data ---
arr = np.array(bird_confs, dtype=float)
q20 = float(np.quantile(arr, 0.20))
q50 = float(np.quantile(arr, 0.50))
q80 = float(np.quantile(arr, 0.80))

# Default suggestions:
SUGG_DETECT = float(np.clip(CONF_BIRD, 0.05, 0.99))
SUGG_TRACK  = float(np.clip(min(q20, 0.6 * CONF_BIRD), 0.05, SUGG_DETECT))  # lower than detect

print(f"From 4.5a: q20={q20:.2f} q50={q50:.2f} q80={q80:.2f}")
print(f"From 4.5b: CONF_BIRD={CONF_BIRD:.2f}")
print(f"Suggested: CONF_DETECT≈{SUGG_DETECT:.2f}  CONF_TRACK≈{SUGG_TRACK:.2f}")

# --- UI / overrides ---
USE_SUGGESTED_THRESHOLDS = True  #@param {type:"boolean"}

CONF_DETECT_MANUAL = 0.25  #@param {type:"number"}
CONF_TRACK_MANUAL  = 0.10  #@param {type:"number"}

# --- Tracking persistence ---
MISS_MAX = 8  #@param {type:"integer"}  # tolerate losing track this many frames

# --- Smoothing / landing logic ---
EMA_ALPHA = 0.35     #@param {type:"number"}  # 0.2..0.5
LAND_SECONDS = 0.35  #@param {type:"number"}  # how long "still" counts as landed
MOVE_PX_THRESH = 20  #@param {type:"integer"} # allowed EMA motion per frame

# --- Output / behavior ---
SAVE_MAX_PER_EPISODE = 3  #@param {type:"integer"}
COOLDOWN_SECONDS = 2.0    #@param {type:"number"}

# --- Apply thresholds ---
if USE_SUGGESTED_THRESHOLDS:
    CONF_DETECT = float(SUGG_DETECT)
    CONF_TRACK  = float(SUGG_TRACK)
    reason = "suggested (based on CONF_BIRD + q20)"
else:
    CONF_DETECT = float(CONF_DETECT_MANUAL)
    CONF_TRACK  = float(CONF_TRACK_MANUAL)
    reason = "manual override"

# Sanity: track must be <= detect
CONF_TRACK = float(min(CONF_TRACK, CONF_DETECT))

LAND_N_FRAMES = max(2, int(round(LAND_SECONDS * fps)))

print("\n✅ Applied 4.5c config:")
print(f"fps≈{fps:.2f}  LAND_SECONDS={LAND_SECONDS:.2f}s -> LAND_N_FRAMES={LAND_N_FRAMES}")
print(f"CONF_DETECT={CONF_DETECT:.2f}  CONF_TRACK={CONF_TRACK:.2f}  ({reason})")
print(f"MISS_MAX={MISS_MAX}  EMA_ALPHA={EMA_ALPHA:.2f}  MOVE_PX_THRESH={MOVE_PX_THRESH}px")
print(f"SAVE_MAX_PER_EPISODE={SAVE_MAX_PER_EPISODE}  COOLDOWN_SECONDS={COOLDOWN_SECONDS:.2f}s")

### Read the tracker as text before you watch the video

The next cell runs the same detect→track→ROI→persistence→trigger loop as the annotated video, but instead of rendering frames it emits a **live transcript**: LOCK ON, ENTER ROI, LANDED, SPRAY EVENT, COOLDOWN, UNLOCK. Watch it stream past.

This is the pedagogical centre of Act 1. A video is passive – the model's decisions flash by at 30 fps. The transcript forces you to read each decision one line at a time. The same reading posture – log forensics – is how IHL and human-rights investigators reconstruct what an autonomous system did after the fact. Hold that thought for Act 3.

In [ ]:
#@title 5b) Run YOLO with lock-on tracking (VERBOSE + tqdm + "would spray" logs)
import os, cv2, math, re, glob, shutil
from ultralytics import YOLO
from tqdm import tqdm

# --- Requirements ---
needed = ["MODEL_NAME","trimmed_video_path","OUT_DIR","ROI_A","ROI_B",
          "CONF_DETECT","CONF_TRACK","MISS_MAX","EMA_ALPHA",
          "LAND_N_FRAMES","MOVE_PX_THRESH","SAVE_MAX_PER_EPISODE","COOLDOWN_SECONDS"]
for v in needed:
    if v not in globals():
        raise NameError(f"Missing `{v}`. Run 4, 4.5a/b, 4.5c first.")

# --- Options ---
VERBOSE = True  #@param {type:"boolean"}
PRINT_EVERY_SEC = 2.0  #@param {type:"number"}  # periodic status line (set 0 to disable)

RESET_SCREENSHOTS_FOLDER = True  #@param {type:"boolean"}

# --- Setup output folder (fresh run) ---
OUT_FRAMES_DIR = os.path.join(OUT_DIR, "screenshots")
if RESET_SCREENSHOTS_FOLDER and os.path.exists(OUT_FRAMES_DIR):
    shutil.rmtree(OUT_FRAMES_DIR)
os.makedirs(OUT_FRAMES_DIR, exist_ok=True)

model = YOLO(MODEL_NAME)

def in_roi(cx, cy, roi):
    x1, y1, x2, y2 = roi
    return x1 <= cx <= x2 and y1 <= cy <= y2

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, ix2-ix1), max(0, iy2-iy1)
    inter = iw * ih
    if inter == 0:
        return 0.0
    a_area = max(0, ax2-ax1) * max(0, ay2-ay1)
    b_area = max(0, bx2-bx1) * max(0, by2-by1)
    denom = a_area + b_area - inter
    return float(inter / denom) if denom > 0 else 0.0

cap = cv2.VideoCapture(trimmed_video_path)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {trimmed_video_path}")

fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
print(f"✅ FPS ~ {fps:.2f} | frames ~ {total_frames}")
print(f"Config: detect≥{CONF_DETECT:.2f} track≥{CONF_TRACK:.2f} | landed={LAND_N_FRAMES} frames | move≤{MOVE_PX_THRESH}px | cooldown={COOLDOWN_SECONDS:.1f}s")

# --- Tracker state ---
locked = False
miss_count = 0
prev_box = None
ema = None
prev_ema = None
still_count = 0
current_roi = None
saved_in_episode = {"A": 0, "B": 0}
last_fire_ts = {"A": -1e9, "B": -1e9}

# logging helpers
last_status_print_t = -1e9
last_roi = None

def ts(frame_idx):  # seconds
    return frame_idx / fps

def log(msg):
    if VERBOSE:
        print(msg)

INFER_CONF = float(CONF_TRACK)

saved_total = 0
frame_idx = 0

pbar = tqdm(total=total_frames if total_frames > 0 else None, desc="Processing frames")

while True:
    ok, frame = cap.read()
    if not ok:
        break
    frame_idx += 1
    if total_frames > 0:
        pbar.update(1)

    # periodic status line
    t_now = ts(frame_idx)
    if PRINT_EVERY_SEC and (t_now - last_status_print_t) >= PRINT_EVERY_SEC:
        last_status_print_t = t_now
        pbar.set_postfix({
            "t(s)": f"{t_now:.1f}",
            "locked": locked,
            "roi": current_roi if current_roi else "-",
            "still": f"{still_count}/{LAND_N_FRAMES}",
            "miss": miss_count
        })

    # --- Detect birds at low threshold ---
    res = model.predict(frame, imgsz=640, conf=INFER_CONF, verbose=False)[0]
    birds = []
    if res.boxes is not None and len(res.boxes) > 0:
        for box in res.boxes:
            cls = int(box.cls[0])
            name = model.names.get(cls, str(cls))
            if name != "bird":
                continue
            conf = float(box.conf[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2
            birds.append((conf, (x1,y1,x2,y2), (cx,cy)))

    chosen = None

    if not locked:
        strong = [b for b in birds if b[0] >= CONF_DETECT]
        if strong:
            chosen = max(strong, key=lambda t: t[0])
            locked = True
            miss_count = 0
            prev_box = chosen[1]
            ema = (float(chosen[2][0]), float(chosen[2][1]))
            prev_ema = None
            still_count = 0
            current_roi = None
            saved_in_episode = {"A": 0, "B": 0}
            last_roi = None
            log(f"[{t_now:6.2f}s] ✅ Bird detected (conf {chosen[0]:.2f}) → LOCK ON")
        else:
            continue
    else:
        if birds:
            best, best_i = None, -1.0
            for b in birds:
                i = iou_xyxy(prev_box, b[1]) if prev_box is not None else 0.0
                if i > best_i:
                    best, best_i = b, i
            chosen = best
            miss_count = 0
        else:
            miss_count += 1
            if miss_count > MISS_MAX:
                log(f"[{t_now:6.2f}s] ⚠️ Lost track for {MISS_MAX} frames → UNLOCK")
                locked = False
                prev_box = None
                ema = None
                prev_ema = None
                still_count = 0
                current_roi = None
                saved_in_episode = {"A": 0, "B": 0}
                last_roi = None
            continue

    conf, box, (cx, cy) = chosen
    x1,y1,x2,y2 = box

    # ROI membership
    roi_now = "A" if in_roi(cx, cy, ROI_A) else ("B" if in_roi(cx, cy, ROI_B) else None)

    # log ROI transitions
    if roi_now != last_roi:
        if roi_now is None and last_roi is not None:
            log(f"[{t_now:6.2f}s] ↩️ Bird left ROI {last_roi} → reset landing episode")
        elif roi_now is not None:
            log(f"[{t_now:6.2f}s] 🎯 Bird entered ROI {roi_now}")
        last_roi = roi_now

    # EMA centroid
    if ema is None:
        ema = (float(cx), float(cy))
    else:
        ex, ey = ema
        ex = EMA_ALPHA * float(cx) + (1 - EMA_ALPHA) * ex
        ey = EMA_ALPHA * float(cy) + (1 - EMA_ALPHA) * ey
        ema = (ex, ey)

    # Landing logic
    if roi_now is None:
        still_count = 0
        prev_ema = None
        current_roi = None
        saved_in_episode = {"A": 0, "B": 0}
    else:
        if current_roi != roi_now:
            still_count = 0
            prev_ema = None
            saved_in_episode[roi_now] = 0
            current_roi = roi_now

        if prev_ema is not None:
            dx = ema[0] - prev_ema[0]
            dy = ema[1] - prev_ema[1]
            dist = math.sqrt(dx*dx + dy*dy)
            still_count = still_count + 1 if dist <= MOVE_PX_THRESH else 0
        prev_ema = ema

        # When it first becomes "landed", announce it
        if still_count == LAND_N_FRAMES:
            log(f"[{t_now:6.2f}s] 🧊 'LANDED' in ROI {roi_now}: still for ≥ {LAND_N_FRAMES} frames (~{LAND_N_FRAMES/fps:.2f}s)")

        if still_count >= LAND_N_FRAMES:
            # cooldown check
            since = t_now - last_fire_ts[roi_now]
            if since >= COOLDOWN_SECONDS:
                # "would spray"
                log(f"[{t_now:6.2f}s] 💦 Would SPRAY ROI {roi_now} now (cooldown OK, landed OK).")

                # save annotated evidence (limited per episode)
                if saved_in_episode[roi_now] < SAVE_MAX_PER_EPISODE:
                    vis = frame.copy()
                    # draw ROIs
                    for rr, lab in [(ROI_A, "ROI A"), (ROI_B, "ROI B")]:
                        rx1, ry1, rx2, ry2 = rr
                        cv2.rectangle(vis, (rx1,ry1), (rx2,ry2), (255,255,255), 2)
                        cv2.putText(vis, lab, (rx1, max(15, ry1-6)),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
                    # draw bird
                    cv2.rectangle(vis, (x1,y1), (x2,y2), (255,255,255), 2)
                    cv2.putText(vis, f"bird {conf:.2f} (detect {CONF_DETECT:.2f}/track {CONF_TRACK:.2f})",
                                (x1, max(15,y1-6)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
                    cv2.putText(vis, f"LANDED ROI {roi_now} still={still_count}/{LAND_N_FRAMES}",
                                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)

                    out_path = os.path.join(OUT_FRAMES_DIR, f"landed_{roi_now}_t{t_now:.2f}_f{frame_idx}.jpg")
                    cv2.imwrite(out_path, vis)
                    saved_in_episode[roi_now] += 1
                    saved_total += 1

                # mark firing (even if we didn't save image due to episode limit)
                last_fire_ts[roi_now] = t_now
            else:
                # occasionally mention cooldown blocking (not every frame)
                if still_count == LAND_N_FRAMES:
                    log(f"[{t_now:6.2f}s] ⏳ Cooldown active for ROI {roi_now}: {since:.2f}s since last fire (need {COOLDOWN_SECONDS:.2f}s)")

    prev_box = box

pbar.close()
cap.release()

print("\n✅ Done.")
print("Saved screenshots:", saved_total)
print("📁 Screenshots folder:", OUT_FRAMES_DIR)
!ls -lh "{OUT_FRAMES_DIR}" | head -n 30

In [ ]:
#@title 5.1) Create annotated video (USES 5b logic) + event banners + centered "SPRAYED" stamp (1s)
import os, cv2, math
from ultralytics import YOLO

# --- Requirements (expects 4 + 4.5a/b + 4.5c already run) ---
req = ["MODEL_NAME","trimmed_video_path","OUT_DIR","ROI_A","ROI_B",
       "CONF_DETECT","CONF_TRACK","MISS_MAX","EMA_ALPHA",
       "LAND_N_FRAMES","MOVE_PX_THRESH","COOLDOWN_SECONDS"]
for var in req:
    if var not in globals():
        raise NameError(f"Missing `{var}`. Run 4, 4.5a/b, 4.5c first.")

annot_path = os.path.join(OUT_DIR, "annotated.mp4")
model = YOLO(MODEL_NAME)

def in_roi(cx, cy, roi):
    x1, y1, x2, y2 = roi
    return x1 <= cx <= x2 and y1 <= cy <= y2

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, ix2-ix1), max(0, iy2-iy1)
    inter = iw * ih
    if inter == 0:
        return 0.0
    a_area = max(0, ax2-ax1) * max(0, ay2-ay1)
    b_area = max(0, bx2-bx1) * max(0, by2-by1)
    denom = a_area + b_area - inter
    return float(inter / denom) if denom > 0 else 0.0

def draw_rois(frame):
    for rr, lab in [(ROI_A, "ROI A"), (ROI_B, "ROI B")]:
        rx1, ry1, rx2, ry2 = rr
        cv2.rectangle(frame, (rx1,ry1), (rx2,ry2), (255,255,255), 2)
        cv2.putText(frame, lab, (rx1, max(15, ry1-8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)

def top_event_banner(frame, text):
    H, W = frame.shape[:2]
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (W, 72), (0,0,0), -1)
    frame[:] = cv2.addWeighted(overlay, 0.55, frame, 0.45, 0)
    cv2.putText(frame, text, (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255,255,255), 4)

def center_stamp(frame, text):
    """Centered big stamp with dark translucent box."""
    H, W = frame.shape[:2]
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale = 3.0
    thick = 10
    (tw, th), _ = cv2.getTextSize(text, font, scale, thick)
    x = (W - tw) // 2
    y = (H + th) // 2

    pad = 30
    overlay = frame.copy()
    cv2.rectangle(overlay, (x - pad, y - th - pad), (x + tw + pad, y + pad), (0,0,0), -1)
    frame[:] = cv2.addWeighted(overlay, 0.55, frame, 0.45, 0)

    cv2.putText(frame, text, (x, y), font, scale, (255,255,255), thick, cv2.LINE_AA)

cap = cv2.VideoCapture(trimmed_video_path)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {trimmed_video_path}")

fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(annot_path, fourcc, fps, (W, H))
if not writer.isOpened():
    cap.release()
    raise RuntimeError("Could not open VideoWriter (mp4v).")

print("Writing annotated video to:", annot_path)
print("Video size:", W, "x", H, "| FPS:", fps)

# --- Banner timing ---
EVENT_HOLD_FRAMES = int(round(0.6 * fps))  # top event banner hold
SPRAY_HOLD_FRAMES = int(round(1.0 * fps))  # centered "SPRAYED" hold (1s)

event_hold = 0
event_text = ""

spray_hold = 0  # centered stamp counter

# --- 5b-style lock-on tracker state ---
locked = False
miss_count = 0
prev_box = None
ema = None
prev_ema = None
still_count = 0
current_roi = None
last_roi = None
last_fire_ts = {"A": -1e9, "B": -1e9}

INFER_CONF = float(CONF_TRACK)

frame_idx = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    frame_idx += 1
    t_now = frame_idx / fps

    # Detect birds at low threshold (track mode)
    res = model.predict(frame, imgsz=640, conf=INFER_CONF, verbose=False)[0]
    birds = []
    if res.boxes is not None and len(res.boxes) > 0:
        for box in res.boxes:
            cls = int(box.cls[0])
            name = model.names.get(cls, str(cls))
            if name != "bird":
                continue
            conf = float(box.conf[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2
            birds.append((conf, (x1,y1,x2,y2), (cx,cy)))

    chosen = None
    if not locked:
        strong = [b for b in birds if b[0] >= CONF_DETECT]
        if strong:
            chosen = max(strong, key=lambda t: t[0])
            locked = True
            miss_count = 0
            prev_box = chosen[1]
            ema = (float(chosen[2][0]), float(chosen[2][1]))
            prev_ema = None
            still_count = 0
            current_roi = None
            last_roi = None

            event_text = f"LOCK ON  t={t_now:.2f}s  conf={chosen[0]:.2f}"
            event_hold = EVENT_HOLD_FRAMES

        draw_rois(frame)
        if event_hold > 0:
            top_event_banner(frame, event_text); event_hold -= 1
        if spray_hold > 0:
            center_stamp(frame, "SPRAYED"); spray_hold -= 1
        writer.write(frame)
        continue

    # LOCKED
    if birds:
        best, best_i = None, -1.0
        for b in birds:
            i = iou_xyxy(prev_box, b[1]) if prev_box is not None else 0.0
            if i > best_i:
                best, best_i = b, i
        chosen = best
        miss_count = 0
    else:
        miss_count += 1
        if miss_count > MISS_MAX:
            locked = False
            prev_box = None
            ema = None
            prev_ema = None
            still_count = 0
            current_roi = None
            last_roi = None

            event_text = f"UNLOCK  t={t_now:.2f}s  lost track for {MISS_MAX} frames"
            event_hold = EVENT_HOLD_FRAMES

        draw_rois(frame)
        if event_hold > 0:
            top_event_banner(frame, event_text); event_hold -= 1
        if spray_hold > 0:
            center_stamp(frame, "SPRAYED"); spray_hold -= 1
        writer.write(frame)
        continue

    conf, box, (cx, cy) = chosen
    x1,y1,x2,y2 = box

    # ROI membership (centroid)
    roi_now = "A" if in_roi(cx, cy, ROI_A) else ("B" if in_roi(cx, cy, ROI_B) else None)

    # ROI transition events
    if roi_now != last_roi:
        if last_roi is not None and roi_now is None:
            event_text = f"LEFT ROI {last_roi}  t={t_now:.2f}s"
            event_hold = EVENT_HOLD_FRAMES
        elif roi_now is not None:
            event_text = f"ENTER ROI {roi_now}  t={t_now:.2f}s"
            event_hold = EVENT_HOLD_FRAMES
        last_roi = roi_now

    # EMA centroid
    if ema is None:
        ema = (float(cx), float(cy))
    else:
        ex, ey = ema
        ex = EMA_ALPHA * float(cx) + (1 - EMA_ALPHA) * ex
        ey = EMA_ALPHA * float(cy) + (1 - EMA_ALPHA) * ey
        ema = (ex, ey)

    # Landing logic
    if roi_now is None:
        still_count = 0
        prev_ema = None
        current_roi = None
    else:
        if current_roi != roi_now:
            still_count = 0
            prev_ema = None
            current_roi = roi_now

        if prev_ema is not None:
            dx = ema[0] - prev_ema[0]
            dy = ema[1] - prev_ema[1]
            dist = math.sqrt(dx*dx + dy*dy)
            still_count = still_count + 1 if dist <= MOVE_PX_THRESH else 0
        prev_ema = ema

        if still_count == LAND_N_FRAMES:
            event_text = f"LANDED ROI {roi_now}  t={t_now:.2f}s  still>={LAND_N_FRAMES} frames"
            event_hold = EVENT_HOLD_FRAMES

        if still_count >= LAND_N_FRAMES:
            since = t_now - last_fire_ts[roi_now]
            if since >= COOLDOWN_SECONDS:
                # SPRAY event
                last_fire_ts[roi_now] = t_now
                spray_hold = SPRAY_HOLD_FRAMES
                event_text = f"SPRAY EVENT  ROI {roi_now}  t={t_now:.2f}s"
                event_hold = EVENT_HOLD_FRAMES
            else:
                if still_count == LAND_N_FRAMES:
                    event_text = f"COOLDOWN ROI {roi_now}  t={t_now:.2f}s  ({since:.2f}s/{COOLDOWN_SECONDS:.2f}s)"
                    event_hold = EVENT_HOLD_FRAMES

    # Draw overlays
    draw_rois(frame)
    cv2.rectangle(frame, (x1,y1), (x2,y2), (255,255,255), 2)
    cv2.putText(frame, f"bird {conf:.2f} (detect {CONF_DETECT:.2f} / track {CONF_TRACK:.2f})",
                (x1, max(15,y1-8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
    cv2.circle(frame, (int(ema[0]), int(ema[1])), 5, (255,255,255), -1)

    # Always-on debug line
    cv2.putText(frame, f"still={still_count}/{LAND_N_FRAMES}  roi={roi_now}  miss={miss_count}",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)

    # Event overlays
    if event_hold > 0:
        top_event_banner(frame, event_text); event_hold -= 1
    if spray_hold > 0:
        center_stamp(frame, "SPRAYED"); spray_hold -= 1

    writer.write(frame)
    prev_box = box

cap.release()
writer.release()
print("Done.")
!ls -lh "{annot_path}"

In [ ]:
#@title 6) Preview: screenshots + web-playable annotated video (H.264)
import os, glob, random
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import Video, display

SHOW_N_IMAGES = 16  #@param {type:"integer"}
SEED = 0           #@param {type:"integer"}

screens_dir = os.path.join(OUT_DIR, "screenshots")
annot_path  = os.path.join(OUT_DIR, "annotated.mp4")
web_path    = os.path.join(OUT_DIR, "annotated_web.mp4")

# ---- 1) Show screenshots ----
paths = sorted(glob.glob(os.path.join(screens_dir, "*.jpg")))
print("📁 Screenshots folder:", screens_dir)
print("Found screenshots:", len(paths))

if paths:
    random.seed(SEED)
    chosen = paths if len(paths) <= SHOW_N_IMAGES else random.sample(paths, SHOW_N_IMAGES)
    cols = 4
    rows = (len(chosen) + cols - 1) // cols
    plt.figure(figsize=(4*cols, 3*rows))
    for i, p in enumerate(chosen, start=1):
        img = Image.open(p)
        plt.subplot(rows, cols, i)
        plt.imshow(img)
        plt.axis("off")
        plt.title(os.path.basename(p), fontsize=9)
    plt.suptitle("Sample TARGETED/LANDED screenshots", fontsize=14)
    plt.show()
else:
    print("⚠️ No screenshots found yet (run 5b to generate them).")

# ---- 2) Re-encode + play annotated video ----
if not os.path.exists(annot_path):
    print("\n⚠️ No annotated.mp4 found. Run Cell 5.1 first.")
else:
    if os.path.exists(web_path):
        os.remove(web_path)
    print("\n🔄 Re-encoding annotated.mp4 → annotated_web.mp4 (H.264 for Colab playback)...")
    !ffmpeg -y -i "{annot_path}" -c:v libx264 -pix_fmt yuv420p -movflags +faststart -crf 23 -preset veryfast -an "{web_path}"
    print("✅ Annotated web video:", web_path)
    display(Video(web_path, embed=True, width=900))

### Debug appendix – only if you saw no LANDED events

If the verbose log and the annotated video produced no `LANDED` or `SPRAY` events, the pipeline's parameters didn't match the footage. Run the cell below to see a frame-by-frame diagnostic: how many frames had birds at all, how many birds entered ROI A vs B, and a few sample frames saved to disk so you can eyeball what's happening. Then tune `CONF_BIRD`, `ROI_A`, `ROI_B`, `MOVE_PX_THRESH`, or `LAND_SECONDS` up in the 4.5 cells and re-run.

In [ ]:
#@title 5d) DEBUG in case "LANDED" is never triggered (counts + save example frames)
import os, cv2, math
from ultralytics import YOLO

dbg_dir = os.path.join(OUT_DIR, "debug_landed")
if os.path.exists(dbg_dir):
    import shutil; shutil.rmtree(dbg_dir)
os.makedirs(dbg_dir, exist_ok=True)

model = YOLO(MODEL_NAME)

def in_roi(cx, cy, roi):
    x1, y1, x2, y2 = roi
    return x1 <= cx <= x2 and y1 <= cy <= y2

def best_bird_det(result):
    best = None
    if result.boxes is None or len(result.boxes) == 0:
        return None
    for box in result.boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        name = model.names.get(cls, str(cls))
        if name != "bird" or conf < CONF_BIRD:
            continue
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        cx = (x1 + x2) // 2
        cy = (y1 + y2) // 2
        if best is None or conf > best[4]:
            best = (x1,y1,x2,y2,conf,cx,cy)
    return best

cap = cv2.VideoCapture(trimmed_video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0

INFER_CONF = min(0.15, CONF_BIRD)

frames = 0
frames_with_bird = 0
frames_in_A = 0
frames_in_B = 0

# stillness counters per ROI (as in Cell 5)
state = {"A": {"still": 0, "prev": None, "saved": 0},
         "B": {"still": 0, "prev": None, "saved": 0}}
frames_still_A = 0
frames_still_B = 0
frames_landed_A = 0
frames_landed_B = 0

saved_examples = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break
    frames += 1

    res = model.predict(frame, imgsz=640, conf=INFER_CONF, verbose=False)[0]
    det = best_bird_det(res)

    if not det:
        continue

    frames_with_bird += 1
    x1,y1,x2,y2,conf,cx,cy = det

    inside_A = in_roi(cx, cy, ROI_A)
    inside_B = in_roi(cx, cy, ROI_B)

    if inside_A: frames_in_A += 1
    if inside_B: frames_in_B += 1

    # update stillness for each ROI separately (only when inside)
    for key, inside in [("A", inside_A), ("B", inside_B)]:
        st = state[key]
        if inside:
            if st["prev"] is not None:
                dx = cx - st["prev"][0]
                dy = cy - st["prev"][1]
                dist = math.sqrt(dx*dx + dy*dy)
                st["still"] = st["still"] + 1 if dist <= MOVE_PX_THRESH else 0
            st["prev"] = (cx,cy)

            if st["still"] > 0:
                if key == "A": frames_still_A += 1
                else: frames_still_B += 1

            if st["still"] >= LAND_N_FRAMES:
                if key == "A": frames_landed_A += 1
                else: frames_landed_B += 1

                # save a few examples of "would have triggered"
                if saved_examples < 10:
                    vis = frame.copy()
                    # draw ROI boxes
                    for roi, label in [(ROI_A, "ROI A"), (ROI_B, "ROI B")]:
                        rx1, ry1, rx2, ry2 = roi
                        cv2.rectangle(vis, (rx1,ry1), (rx2,ry2), (255,255,255), 2)
                        cv2.putText(vis, label, (rx1, max(15, ry1-6)),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
                    # draw bird box
                    cv2.rectangle(vis, (x1,y1), (x2,y2), (255,255,255), 2)
                    cv2.putText(vis, f"bird {conf:.2f} | inside {key} | still={st['still']}",
                                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
                    out = os.path.join(dbg_dir, f"would_trigger_{key}_f{frames}.jpg")
                    cv2.imwrite(out, vis)
                    saved_examples += 1
        else:
            st["still"] = 0
            st["prev"] = None

cap.release()

print("Frames total:", frames)
print("Frames with bird (>= CONF_BIRD):", frames_with_bird)
print("Frames bird centroid in ROI_A:", frames_in_A)
print("Frames bird centroid in ROI_B:", frames_in_B)
print("Frames with stillness>0 in A/B:", frames_still_A, "/", frames_still_B)
print("Frames meeting LANDED in A/B:", frames_landed_A, "/", frames_landed_B)
print("Saved example triggers (if any):", saved_examples)
print("Debug folder:", dbg_dir)
!ls -lh "{dbg_dir}" | head -n 30

### What just happened

You ran a complete autonomous-action pipeline:

1. **Detect.** YOLO scans every frame for objects of class `bird`. Each detection comes with a confidence score and bounding box.
2. **Lock on.** Once confidence exceeds `CONF_DETECT`, the tracker locks. From then on it follows the same target at the lower `CONF_TRACK` threshold (so the lock persists through brief drops in confidence).
3. **Region of interest.** A bird inside one of two named ROIs (A, B) is treated as "approaching the landing zone."
4. **Persistence.** A smoothed centroid (EMA) tracks movement. If the bird stays still for `LAND_N_FRAMES` frames inside the ROI, it counts as landed.
5. **Trigger.** Landed bird + cooldown elapsed → "would spray" event. The system writes a screenshot, draws a SPRAYED banner, and waits for cooldown before firing again.

Every parameter you set – `CONF_DETECT`, `CONF_TRACK`, `LAND_N_FRAMES`, `MOVE_PX_THRESH`, `COOLDOWN_SECONDS` – is a knob a designer chose. None is required by the world; all are choices. Hold that thought.

# Act 2 // The Historical Echo

In 1943, B.F. Skinner – better known for pigeon-pecking experiments and operant conditioning – proposed and prototyped **Project Pigeon**, a pigeon-guided missile. A live pigeon, harnessed inside the nose cone of a glide bomb, would peck at an image of the target ship projected onto a screen in front of it. The peck location, mechanically converted into steering commands, would correct the missile's course toward the target. Skinner trained pigeons to peck reliably at silhouettes of warships. The National Defense Research Committee funded the work to the tune of $25,000 (~$450,000 in 2026 dollars).

The project was prototyped, demonstrated to military reviewers, and cancelled in 1944 – not on ethical grounds, but because radar-guided munitions matured first. Skinner kept the pigeons. The trained birds reportedly retained their target-discrimination ability for years.

The structural template is identical to what you just built. A trained classifier (a pigeon, a YOLO model) detects a target by visual pattern-match. A controller (a bell-crank linkage, a tracker state machine) converts the classifier's output into a continuous lock. Persistence in the target window triggers an action (course correction, spray event). The classifier is opaque to the operator: nobody can interrogate why a pigeon pecked where it did, just as nobody can explain a YOLO confidence score from first principles. The operator authorises the system, points it at a region, and trusts the loop.

**This is not a new idea.**

# Act 3 // Structural Identity

Change the target class from `bird` to `person`. Change the trigger from "spray water" to "recommend airstrike." Change the scale from one balcony to a city. Change the cooldown from two seconds to twenty seconds of human review.

The detection-track-ROI-persistence-trigger code does not change. That is the point of this act.

In April 2024, +972 Magazine and Local Call reported that the Israeli military used a system called **Lavender** to generate targeting recommendations during the war in Gaza. By the reports' account, Lavender scored individuals on the basis of signals-intelligence features and produced names – tens of thousands of them – marked for strike. Operators had on the order of twenty seconds per recommendation to approve or reject. The system was treated, by accounts of those operating it, as authoritative.

Lavender's classifier worked over different features – communications patterns, network-graph proximity, device movements – not images. But the structural template is the same as what you ran in Act 1. **Detect → score → threshold → action.** Below, we run the *same code* over a street scene with `person` as the target class. The output is no different in form. Confidence scores, bounding boxes, threshold-based filtering. What changes is what the operator does next.

In [ ]:
#@title Same pipeline, target class = person
#@markdown Run YOLO over a street scene. Filter for the COCO 'person' class.
#@markdown The detection logic is identical to Act 1 – only the class label changes.

import urllib.request
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
import numpy as np

model_p = YOLO("yolov8n.pt")  # same model used in Act 1

# Public domain street scene
PERSON_IMG_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/f/fa/Crowd_fills_street_at_a_May_2018_protest.jpg/1000px-Crowd_fills_street_at_a_May_2018_protest.jpg"
PERSON_IMG_PATH = "/tmp/street_scene_persons.jpg"

try:
    _req = urllib.request.Request(PERSON_IMG_URL, headers={"User-Agent": "IntlLawAI-Lab/1.0 (https://github.com/niccoloridi/International-Law-and-AI)"})
    with urllib.request.urlopen(_req, timeout=15) as _r, open(PERSON_IMG_PATH, "wb") as _f:
        _f.write(_r.read())
except Exception as e:
    # Fallback to a different public domain crowd image
    PERSON_IMG_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/1/1c/July_30_Bike_Protest_crowd_-_safe_streets_now.jpg/1024px-July_30_Bike_Protest_crowd_-_safe_streets_now.jpg"
    _req = urllib.request.Request(PERSON_IMG_URL, headers={"User-Agent": "IntlLawAI-Lab/1.0 (https://github.com/niccoloridi/International-Law-and-AI)"})
    with urllib.request.urlopen(_req, timeout=15) as _r, open(PERSON_IMG_PATH, "wb") as _f:
        _f.write(_r.read())

CONF_PERSON = 0.30  #@param {type:"number"}

results_p = model_p.predict(PERSON_IMG_PATH, conf=CONF_PERSON, verbose=False)
boxes = results_p[0].boxes
person_dets = []
for box in boxes:
    cls = int(box.cls[0])
    if model_p.names.get(cls) == "person":
        person_dets.append({
            "conf": float(box.conf[0]),
            "bbox": [int(v) for v in box.xyxy[0].tolist()],
        })

print(f"Detected {len(person_dets)} persons at conf >= {CONF_PERSON}")
print()
print("Top 5 by confidence:")
for d in sorted(person_dets, key=lambda x: -x["conf"])[:5]:
    print(f"  person  conf={d['conf']:.3f}  bbox={d['bbox']}")

img = cv2.imread(PERSON_IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
for d in person_dets:
    x1, y1, x2, y2 = d["bbox"]
    cv2.rectangle(img_rgb, (x1, y1), (x2, y2), (255, 64, 64), 2)
    cv2.putText(img_rgb, f"person {d['conf']:.2f}", (x1, max(15, y1 - 6)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 64, 64), 2)

plt.figure(figsize=(12, 8))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(f"Same code as Act 1. Class = person. Threshold = {CONF_PERSON}")
plt.show()

### Notice what didn't change

- The model: same YOLOv8n weights.
- The detection code: a `model.predict()` call with a confidence threshold.
- The output: bounding boxes and confidence scores.
- The threshold question: where do you set the cutoff between "match" and "no match"?

What changed is the consequence of acting on the output. In Act 1, a false positive means a confused pigeon and a wet windowsill. In Act 3 framing, a false positive means a wrongful target.

The tracker, ROI logic, EMA smoothing, and trigger machinery from Act 1 would all transfer unchanged to a person-targeting variant. The only structural difference between a pigeon-deterrent and a Lavender-style system is the choice of target class and the consequences attached to the trigger event.

## The precision-recall tradeoff at scale

A classifier emits a number; a threshold turns the number into an action. Set the threshold low and you catch more targets at the cost of more false positives. Set it high and you miss more targets but get cleaner positives. **There is no "right" threshold from inside the model – only a choice with consequences.**

In [ ]:
#@title ## Precision and Recall: The Tradeoff
#@markdown Calculate precision and recall across thresholds. Show the tradeoff.
import numpy as np
import matplotlib.pyplot as plt



# Ensure YOLO model and a working image path are available, whether Act 1
# ran or not. Uses PERSON_IMG_PATH from the Act-3 download which is the
# pipeline's canonical test image.
from ultralytics import YOLO
if "model" not in globals() or model is None:
    model = YOLO("yolov8n.pt")
if "img_path" not in globals() or not img_path:
    img_path = PERSON_IMG_PATH

# Use the street scene image for detailed threshold analysis
thresholds_detail = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

precision_sim = []  # Simulated precision (based on detection confidence)
recall_sim = []     # Simulated recall (higher threshold = lower recall)

for threshold in thresholds_detail:
    results = model.predict(img_path, conf=threshold, verbose=False)
    detections = results[0].boxes
    
    # Simulated precision: average confidence at this threshold
    if len(detections) > 0:
        avg_conf = np.mean([float(box.conf[0].item()) for box in detections])
        precision_sim.append(avg_conf)
    else:
        precision_sim.append(threshold)  # Lower threshold → lower confidence
    
    # Simulated recall: proportion of detections relative to minimum threshold
    recall = len(detections) / max(1, len(model.predict(img_path, conf=0.05, verbose=False)[0].boxes))
    recall_sim.append(recall)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Precision vs. Threshold
ax1.plot(thresholds_detail, precision_sim, 'o-', linewidth=2, markersize=8, color='green')
ax1.set_xlabel('Confidence Threshold', fontsize=11)
ax1.set_ylabel('Average Detection Confidence', fontsize=11)
ax1.set_title('Precision vs. Threshold', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.set_ylim([0, 1])

# Plot 2: Precision-Recall Tradeoff
ax2.plot(recall_sim, precision_sim, 'o-', linewidth=2, markersize=8, color='blue')
for i, threshold in enumerate(thresholds_detail[::2]):
    ax2.annotate(f'{threshold:.1f}', 
                xy=(recall_sim[i*2], precision_sim[i*2]),
                xytext=(5, 5), textcoords='offset points', fontsize=9)
ax2.set_xlabel('Recall (Detection Rate)', fontsize=11)
ax2.set_ylabel('Precision (Confidence)', fontsize=11)
ax2.set_title('Precision-Recall Tradeoff', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_xlim([0, 1.05])
ax2.set_ylim([0, 1])

plt.suptitle('How Threshold Affects Detection Quality Metrics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey insight: Higher threshold = higher precision, lower recall")
print("             Lower threshold = lower precision, higher recall")
print()
print("In military targeting:")
print("  - High threshold → miss combatants (military failure)")
print("  - Low threshold → target civilians (war crime)")
print("  - No threshold is 'safe' — there is always a tradeoff")

## The Human in the Loop

International discussions on autonomous weapons systems (CCW, UN, ICRC) emphasize **"meaningful human control"**:

- A human must retain the ability to understand and override system decisions
- The human must have adequate time and information to make informed judgments
- The human must have real authority – not just a rubber-stamp role

### The Lavender Problem

According to reports, Lavender:
- Generated ~37 targeting recommendations per day
- Operators had ~20 seconds per recommendation to review
- Operators could not independently verify the scoring logic
- Operators could not interrogate the data underlying each score
- Overriding a recommendation required explicit action; accepting it was default

**Question**: Was this "meaningful human control"?

Most legal experts and the ICRC would answer: **No.** The volume of targets, time pressure, and opacity of the scoring system meant the human operator was a rubber stamp, not a genuine decision-maker.

In [ ]:
#@title ## Simulation: Human Review at Scale
#@markdown Simulate the operator experience: can you meaningfully review targets in 20 seconds?
import numpy as np
import matplotlib.pyplot as plt


# Simulate a day of Lavender operations
np.random.seed(42)
daily_targets = np.random.randint(10, 60, size=30)  # 30 "days" of operations
daily_scores = np.random.randint(30, 95, size=30)   # Random threat scores

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Daily target volumes
ax1.bar(range(1, 31), daily_targets, color='red', alpha=0.6, edgecolor='darkred', linewidth=1.5)
ax1.axhline(y=37, color='black', linestyle='--', linewidth=2, label='Reported Lavender average (37/day)')
ax1.set_xlabel('Day', fontsize=11)
ax1.set_ylabel('Targeting Recommendations', fontsize=11)
ax1.set_title('Daily Volume of Targeting Recommendations', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, axis='y')

# Plot 2: Review time available
total_targets = daily_targets.sum()
total_seconds_available = 24 * 60 * 60  # 1 day in seconds
seconds_per_target = total_seconds_available / total_targets

ax2.barh([f'Day {i+1}' for i in range(5)], [seconds_per_target] * 5, 
         color='orange', alpha=0.6, edgecolor='darkorange', linewidth=1.5)
ax2.axvline(x=20, color='red', linestyle='--', linewidth=2, label='Reported review time (20 sec)')
ax2.set_xlabel('Seconds Available per Target', fontsize=11)
ax2.set_title('Review Time Available (if evenly distributed)', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis='x')

plt.suptitle('The Scaling Problem: Can Humans Meaningfully Review at Volume?', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nSimulation Results:")
print(f"Total targets (30 days): {total_targets}")
print(f"Average targets per day: {total_targets / 30:.1f}")
print(f"\nIf evenly distributed:")
print(f"  Seconds per target: {seconds_per_target:.1f}")
print(f"\nReality (according to reports):")
print(f"  Reported time per target: ~20 seconds")
print(f"  Operators receive ~37 targets per day")
print(f"  Total time available: ~12 minutes per day of actual review")
print(f"  But operators worked 8+ hour shifts...")
print(f"\nConclusion: At this volume and time pressure,")
print(f"meaningful individual review of each target is practically impossible.")

## What "meaningful human control" requires

The ICRC and most international-humanitarian-law scholars agree that "meaningful human control" requires, at minimum:

- **Transparency**: the operator can interrogate why the system made a recommendation.
- **Time**: the operator has enough time to think about each individual case.
- **Authority**: the operator can override the system, and overriding is as easy as accepting.
- **Accountability**: each decision is logged with reasoning, and someone is answerable.

By the reports of those who operated it, Lavender provided few or none of these. Twenty-second review windows, opaque scoring logic, accept-by-default flow, and volume that exceeded any human capacity to evaluate individually.

The pigeon pipeline you ran in Act 1 has the same architectural property. The "would spray" event fires from the loop's logic, not from a deliberative human choice. For pigeons, that is fine. For people, it is the central question.

In [ ]:
#@title ## Your Call: Where Do YOU Set the Threshold?
#@markdown Scenario: You are a human operator reviewing targeting recommendations from an automated system.

print("\n" + "="*80)
print("SCENARIO: AUTONOMOUS TARGETING SYSTEM REVIEW")
print("="*80)

scenario = """
You are a military officer reviewing a targeting recommendation from an automated system.

The system has flagged a person with a threat score of 72 out of 100.
The threshold for authorization is set at 65.

You have 20 seconds to make a decision.

What you know:
- The system detected a person in a surveillance feed
- ML model assigned threat score: 72 (above threshold of 65 → recommendation for strike)
- You cannot see the underlying data the model used (system is proprietary)
- You cannot interrogate the model about its reasoning
- If you reject this target, 100 more are waiting for review today
- The person is in a residential area
- Weather conditions were poor during surveillance (low light, rain)

IHL Obligations:
- Principle of DISTINCTION: Must distinguish combatants from civilians
- Principle of PRECAUTION: Must do everything feasible to minimize civilian harm
- Principle of PROPORTIONALITY: Military advantage vs. civilian harm

"""

print(scenario)
print("="*80)
print("\nQUESTION: What is the MINIMUM confidence threshold that would allow you")
print("to authorize this strike while respecting your IHL obligations?")
print()
print("Consider:")
print("  - What confidence level = 'definitely a combatant'?")
print("  - What confidence level = 'probably a combatant'?")
print("  - What if 10% are false positives (innocents)?")
print("  - What if 20% are false positives?")
print()
print("This is not a technical question. This is a LEGAL question.")
print("And your answer to it is embedded in a single parameter.")

## Summary: What You've Built

- **Act 1**: A working autonomous-action pipeline. YOLO + ROI + persistence + cooldown → SPRAY events on real video. Pigeons.
- **Act 2**: Project Pigeon. The same template, eighty years earlier. The trained-organism-as-classifier in a kill chain is not a new architecture; it is an old one that we have learned to build out of weights instead of feathers.
- **Act 3**: The same pipeline, with the target class swapped. Detection → score → threshold → action. A pigeon deterrent and a Lavender-style targeting system differ in the consequences attached to the trigger, not in the structural form of the loop.

**The threshold is where engineering meets law.** Every parameter in this notebook – `CONF_DETECT`, `LAND_N_FRAMES`, `COOLDOWN_SECONDS`, the choice of target class – is a setting that someone, somewhere, chose. International humanitarian law's principles of distinction, proportionality, and precaution demand that those choices be auditable, justifiable, and reviewable. A pipeline whose human-in-the-loop step takes twenty seconds at industrial scale fails that demand.

The discomfort you should feel, having built the pigeon version yourself, is that the engineering effort to build the other version is small. The law is what stands between the two.

## Before the debrief: what to take away

You have just ported a working pigeon-deterrent pipeline – YOLO detection, ROI check, persistence counter, trigger event – and then swapped the target class from bird to person. The detection-score-threshold loop did not change. Operation Politely Discourage Pigeons and a Lavender-style targeting system differ in the consequences attached to the trigger, not in the structural form of the loop. You also saw the precision-recall tradeoff and the human-review-at-scale simulation: at industrial volume with twenty-second windows, meaningful individual review is an architectural impossibility. The debrief will ask where engineering parameters quietly resolve IHL doctrinal questions that IHL never authorised engineers to settle.

In [ ]:
#@title For your paper
#@markdown One open-ended prompt. Your answer saves to a file you can download and use in your 5,000-word paper.

prompt = """The Lavender system reportedly had an accuracy threshold its operators could tune. Write 300 words arguing whether IHL requires that threshold to be disclosed, to whom, and under what framework. Engage with either Spadaro or the ICRC position."""
print(prompt)
print()

paper_note = "" #@param {type:"string"}

if paper_note.strip():
    _append_to_notes(f"For the paper – Lab 06", paper_note)
    print(f"\nSaved to /content/paper_notes_lab_06.md – download from the Files panel (folder icon, left).")
else:
    print("\n(blank – nothing saved yet. Type your answer in the field above and re-run this cell.)")

## Debrief questions

Your lecturer will run a 35-minute structured discussion after the Colab. Before the debrief, think through these four questions. You do not need to write answers, but come prepared to speak.

1. At what confidence threshold did you set the pivot from bird to person? What convinced you?
2. When you ran the same pipeline on real footage versus your own test images, did detection quality differ enough to matter legally?
3. What did Skinner's Project Pigeon and Lavender have in common structurally? What did they have in common morally?
4. If the ICRC called you as an expert witness, how would you characterise the relationship between YOLO confidence scores and the IHL principle of distinction?